##### 01 - Bronze Layer Ingestion

 The Bronze layer stores raw data exactly as received from the source,
 with added audit metadata. No transformations are applied.

 **Principles:**
 - Schema-on-read (infer schema from source)
 - Append audit columns (_ingestion_timestamp, _source_file, _batch_id)
 - Full reload pattern (idempotent via overwrite)
 - Delta format for ACID guarantees


In [0]:
from pyspark.sql import functions as F
from datetime import datetime
import uuid

BATCH_ID = str(uuid.uuid4())[:8]
INGESTION_TS = datetime.now()

print(f"Batch ID:  {BATCH_ID}")
print(f"Timestamp: {INGESTION_TS}")


Batch ID:  b31242bd
Timestamp: 2026-05-01 06:31:25.644792


##### 1. Ingest Customers (Landing -> Bronze Delta)

In [0]:
df_customers_raw = spark.table("ecommerce_landing.customers")

df_customers_bronze = (
    df_customers_raw
    .withColumn("_ingestion_timestamp", F.lit(INGESTION_TS))
    .withColumn("_source_file", F.lit("ecommerce_landing.customers"))
    .withColumn("_batch_id", F.lit(BATCH_ID))
)

(
    df_customers_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("ecommerce_bronze.customers")
)

print(f"Bronze customers: {df_customers_bronze.count()} rows")
df_customers_bronze.printSchema()


Bronze customers: 10000 rows
root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- signup_date: string (nullable = true)
 |-- status: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = false)
 |-- _source_file: string (nullable = false)
 |-- _batch_id: string (nullable = false)



##### 2. Ingest Products (Landing -> Bronze Delta)

In [0]:
df_products_raw = spark.table("ecommerce_landing.products")

df_products_bronze = (
    df_products_raw
    .withColumn("_ingestion_timestamp", F.lit(INGESTION_TS))
    .withColumn("_source_file", F.lit("ecommerce_landing.products"))
    .withColumn("_batch_id", F.lit(BATCH_ID))
)

(
    df_products_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("ecommerce_bronze.products")
)

print(f"Bronze products: {df_products_bronze.count()} rows")

Bronze products: 200 rows


##### 3. Ingest Orders (Landing -> Bronze Delta)

In [0]:
df_orders_raw = spark.table("ecommerce_landing.orders")

df_orders_bronze = (
    df_orders_raw
    .withColumn("_ingestion_timestamp", F.lit(INGESTION_TS))
    .withColumn("_source_file", F.lit("ecommerce_landing.orders"))
    .withColumn("_batch_id", F.lit(BATCH_ID))
)

(
    df_orders_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("ecommerce_bronze.orders")
)

print(f"Bronze orders: {df_orders_bronze.count()} rows")

Bronze orders: 50255 rows


##### 4. Ingest Order Items (Landing -> Bronze Delta)

In [0]:
df_items_raw = spark.table("ecommerce_landing.order_items")

df_items_bronze = (
    df_items_raw
    .withColumn("_ingestion_timestamp", F.lit(INGESTION_TS))
    .withColumn("_source_file", F.lit("ecommerce_landing.order_items"))
    .withColumn("_batch_id", F.lit(BATCH_ID))
)

(
    df_items_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("ecommerce_bronze.order_items")
)

print(f"Bronze order_items: {df_items_bronze.count()} rows")


Bronze order_items: 105115 rows


##### 5. Bronze Layer Summary

In [0]:
 %sql
SELECT 'customers' AS table_name, COUNT(*) AS row_count FROM ecommerce_bronze.customers
UNION ALL
SELECT 'products', COUNT(*) FROM ecommerce_bronze.products
UNION ALL
SELECT 'orders', COUNT(*) FROM ecommerce_bronze.orders
UNION ALL
SELECT 'order_items', COUNT(*) FROM ecommerce_bronze.order_items
ORDER BY table_name

table_name,row_count
customers,10000
order_items,105115
orders,50255
products,200


##### 6. Verify Delta Table History
Delta Lake maintains a full transaction log. This is useful for auditing.

In [0]:
from delta.tables import DeltaTable

for table in ["customers", "products", "orders", "order_items"]:
    dt = DeltaTable.forName(spark, f"ecommerce_bronze.{table}")
    print(f"\n--- {table} ---")
    dt.history(3).select("version", "timestamp", "operation", "operationMetrics").show(truncate=False)

# COMMAND ----------

print("Bronze ingestion complete! Proceed to notebook 02_silver_transformation.")



--- customers ---
+-------+-------------------+---------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp          |operation                        |operationMetrics                                                                                                                             |
+-------+-------------------+---------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------+
|0      |2026-05-01 06:32:04|CREATE OR REPLACE TABLE AS SELECT|{numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 10000, numOutputBytes -> 181310}|
+-------+-------------------+---------------------------------+----------------------------------------------------------------------------------------------